# Agentic AI on AWS: Multi-Agent Insurance Claims System

This notebook is Part B of the K21Academy lab. You will:

1. Store your Claude API key in **AWS Secrets Manager** and keep claim history in **Amazon S3**.
2. Build the 8 agentic patterns step by step. Each pattern is saved as a small Python file in the `deploy/` folder and then run, so **the code you test here is exactly the code you deploy**.
3. Run the full pipeline on several claims.
4. Build a Docker image with **AWS CodeBuild** (`sm-docker`), push it to **Amazon ECR** and run the web app on **Amazon EC2**.
5. Delete every resource you created.

**Run the cells in order, top to bottom.** If you restart the kernel, run Step 2 again before continuing.
If you stop or restart the JupyterLab space, run Step 1 and Step 2 again: a space restart removes packages installed with `%pip`.

# Part 1: Setup

## Step 1: Install packages
Installs the Anthropic SDK, boto3 and the `sm-docker` command used in Step 17.
Run this every time you start or restart the space.

In [ ]:
%pip install -q anthropic boto3 sagemaker-studio-image-build

## Step 2: Settings
All names used in this notebook are set here. The Region comes from your SageMaker domain.

In [ ]:
import os, sys, json, time, importlib
import boto3
from botocore.exceptions import ClientError

try:
    import anthropic
except ImportError:
    raise SystemExit("The 'anthropic' package is not installed in this kernel. Run the Step 1 cell first, "
                     "then run this cell again. (Restarting the space removes packages installed with %pip.)")

REGION      = boto3.session.Session().region_name or "us-east-1"
ACCOUNT_ID  = boto3.client("sts").get_caller_identity()["Account"]
SECRET_NAME = "insurance-claims/anthropic-api-key"
BUCKET      = f"insurance-claims-history-{ACCOUNT_ID}"
ECR_REPO    = "insurance-claims-ui"
IMAGE_URI   = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO}:latest"
ROLE_NAME   = "EC2InsuranceClaimsRole"
SG_NAME     = "insurance-claims-ui-sg"
MODEL       = "claude-sonnet-4-6"
os.environ["CLAUDE_MODEL"] = MODEL

# Every pattern is written into this folder, then imported from it
os.makedirs("deploy", exist_ok=True)
if "deploy" not in sys.path:
    sys.path.insert(0, "deploy")

def load(name):
    """Import (or re-import after an edit) a module from the deploy folder."""
    importlib.invalidate_caches()
    return importlib.reload(importlib.import_module(name))

print("Region:    ", REGION)
print("Account:   ", ACCOUNT_ID)
print("S3 bucket: ", BUCKET)
print("Image URI: ", IMAGE_URI)

## Step 3: Store your Claude API key in Secrets Manager
Paste your key when prompted. `getpass` hides it, and it is never saved in this notebook.
The cell creates the secret the first time and updates it if it already exists.

In [ ]:
from getpass import getpass

sm = boto3.client("secretsmanager", region_name=REGION)
key = getpass("Paste your Anthropic API key (starts with sk-ant-): ").strip()
assert key.startswith("sk-ant-"), "That does not look like an Anthropic API key. Copy it again from the Claude Console."
value = json.dumps({"ANTHROPIC_API_KEY": key})

try:
    sm.describe_secret(SecretId=SECRET_NAME)
    sm.put_secret_value(SecretId=SECRET_NAME, SecretString=value)
    print("Secret already existed: stored your key as a new version")
except sm.exceptions.ResourceNotFoundException:
    sm.create_secret(Name=SECRET_NAME, SecretString=value,
                     Description="Anthropic API key for the K21 insurance claims lab")
    print("Secret created:", SECRET_NAME)
del key, value

## Step 4: Shared helpers and the Claude client
`common.py` reads the key from Secrets Manager and gives every pattern the same `ask_claude()` and `parse_json()` helpers.

In [ ]:
%%writefile deploy/common.py
"""Shared helpers: the Claude client (API key read from Secrets Manager) and JSON parsing."""
import json
import os
import re

import anthropic
import boto3

REGION = os.environ.get("AWS_REGION") or boto3.session.Session().region_name or "us-east-1"
SECRET_NAME = os.environ.get("SECRET_NAME", "insurance-claims/anthropic-api-key")
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-4-6")

_client = None


def get_client():
    """Create the Claude client once. The key never appears in code or in the notebook."""
    global _client
    if _client is None:
        sm = boto3.client("secretsmanager", region_name=REGION)
        secret = json.loads(sm.get_secret_value(SecretId=SECRET_NAME)["SecretString"])
        _client = anthropic.Anthropic(api_key=secret["ANTHROPIC_API_KEY"])
    return _client


def ask_claude(prompt, max_tokens=800):
    """One prompt in, Claude's text out."""
    response = get_client().messages.create(
        model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}])
    return "".join(b.text for b in response.content if b.type == "text")


def parse_json(text):
    """Return the first JSON object in Claude's reply (handles ```json fences), or None."""
    if not text:
        return None
    fenced = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, re.DOTALL)
    if fenced:
        text = fenced.group(1)
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None

In [ ]:
common = load("common")
client = common.get_client()
print("Key loaded from Secrets Manager: True")
print("Claude client ready. Model:", common.MODEL)

## Step 5: Your first agent
One Claude call returns a decision, a risk level and a reason as JSON.

In [ ]:
%%writefile deploy/basic_agent.py
"""Single agent: one Claude call returns a decision, risk level and reason as JSON."""
from common import ask_claude, parse_json


def analyze_claim(claim_text):
    prompt = f"""You are an insurance claims triage agent.
Return ONLY JSON: {{"decision": "approve | deny | escalate",
"risk_level": "low | medium | high", "reason": "one sentence"}}

Claim: {claim_text}"""
    return parse_json(ask_claude(prompt, 300)) or {"decision": "escalate", "risk_level": "high",
                                                   "reason": "Reply was not valid JSON"}

In [ ]:
basic = load("basic_agent")
sample_claim = ("Rear-end collision, $3,200 in vehicle damage, no injuries, "
                "policy active for 2 years, no prior claims.")
first_result = basic.analyze_claim(sample_claim)
print("Claim:", sample_claim)
print(json.dumps(first_result, indent=2))

## Step 6: Create the S3 bucket for claim memory (Pattern 8: Stateful)
The bucket name ends with your account ID, so it is unique. The `S3ClaimMemory` class saves every result
and can spot a claim that was already submitted. Safe to re-run.

In [ ]:
s3 = boto3.client("s3", region_name=REGION)
try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET)
    else:
        s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})
    print("Created bucket:", BUCKET)
except s3.exceptions.BucketAlreadyOwnedByYou:
    print("Bucket already exists and is yours:", BUCKET)

In [ ]:
%%writefile deploy/memory.py
"""Pattern 8 - Stateful memory in Amazon S3: every result is saved and checked on the next claim."""
import hashlib
import json
import os
from datetime import datetime, timezone

import boto3


def default_bucket():
    account = boto3.client("sts").get_caller_identity()["Account"]
    return os.environ.get("CLAIMS_BUCKET") or f"insurance-claims-history-{account}"


def text_hash(claim_text):
    return hashlib.sha256(" ".join(claim_text.lower().split()).encode()).hexdigest()[:16]


class S3ClaimMemory:
    def __init__(self, bucket=None, prefix="claims/"):
        self.s3 = boto3.client("s3")
        self.bucket = bucket or default_bucket()
        self.prefix = prefix

    def save_claim(self, claim_id, record):
        record = dict(record, saved_at=datetime.now(timezone.utc).isoformat(timespec="seconds"))
        key = f"{self.prefix}{claim_id}.json"
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=json.dumps(record, default=str).encode())
        return key

    def load_claim(self, claim_id):
        obj = self.s3.get_object(Bucket=self.bucket, Key=f"{self.prefix}{claim_id}.json")
        return json.loads(obj["Body"].read())

    def list_claims(self):
        keys = []
        for page in self.s3.get_paginator("list_objects_v2").paginate(Bucket=self.bucket, Prefix=self.prefix):
            keys += [o["Key"] for o in page.get("Contents", [])]
        return keys

    def find_duplicate(self, claim_text, claim_id):
        """Return the ID of an earlier claim with the same text, or None."""
        wanted = text_hash(claim_text)
        for key in self.list_claims():
            other_id = key[len(self.prefix):-5]
            if other_id != claim_id and self.load_claim(other_id).get("text_hash") == wanted:
                return other_id
        return None

In [ ]:
memory = load("memory")
mem = memory.S3ClaimMemory(BUCKET)
key = mem.save_claim("claim_001", {"claim_text": sample_claim, "analysis": first_result,
                                   "text_hash": memory.text_hash(sample_claim)})
print("Saved to:", key)
print("Read back:", json.dumps(mem.load_claim("claim_001"), indent=2))

# Part 2: The agentic patterns

## Step 7: Extract claim facts (structured output)
Claims arrive as free text. Claude turns the text into facts (type, amount, policy age, prior claims)
that the rule-based patterns can check reliably.

In [ ]:
%%writefile deploy/facts.py
"""Turn a free-text claim into structured facts that the rules can check."""
from common import ask_claude, parse_json

DEFAULT_FACTS = {"claim_type": "other", "amount": None, "damage_type": "unknown",
                 "policy_age_months": None, "prior_claims": None,
                 "injuries": False, "cause_stated": False}


def extract_facts(claim_text):
    prompt = f"""Extract the facts from this insurance claim. Use null when a fact is not stated.
Convert years to months (2 years = 24). Return ONLY JSON with exactly these keys:
{{"claim_type": "auto | property | health | liability | other",
  "amount": number or null,
  "damage_type": "short phrase, e.g. rear-end collision, water damage",
  "policy_age_months": number or null,
  "prior_claims": number or null,
  "injuries": true or false,
  "cause_stated": true or false}}

Claim: {claim_text}"""
    facts = dict(DEFAULT_FACTS)
    facts.update(parse_json(ask_claude(prompt, 300)) or {})
    facts["claim_type"] = str(facts["claim_type"]).lower()
    return facts

In [ ]:
facts_mod = load("facts")
sample_facts = facts_mod.extract_facts(sample_claim)
print(json.dumps(sample_facts, indent=2))

## Step 8: Pattern 1, Planner
Breaks the claim into a step-by-step plan before acting.

In [ ]:
%%writefile deploy/planner.py
"""Pattern 1 - Planner: break the claim into a step-by-step processing plan."""
import json
from common import ask_claude, parse_json


def plan_claim(claim_text, facts):
    prompt = f"""You are a claims processing manager. Create a 3-6 step plan to process this claim.
Return ONLY JSON: {{"steps": [{{"step": 1, "action": "...", "why": "..."}}]}}

Claim: {claim_text}
Facts: {json.dumps(facts)}"""
    plan = parse_json(ask_claude(prompt, 600))
    if not plan or not plan.get("steps"):
        plan = {"steps": [{"step": 1, "action": "Verify policy and coverage", "why": "fallback plan"},
                          {"step": 2, "action": "Assess damage and documents", "why": "fallback plan"},
                          {"step": 3, "action": "Decide and notify", "why": "fallback plan"}]}
    return plan

In [ ]:
planner = load("planner")
plan = planner.plan_claim(sample_claim, sample_facts)
for s in plan["steps"]:
    print(f"Step {s['step']}: {s['action']}  ({s.get('why', '')})")

## Step 9: Pattern 2, Router
Picks a processing lane. If Claude's answer is not a valid lane, a simple amount rule is used.

In [ ]:
%%writefile deploy/router.py
"""Pattern 2 - Router: choose a processing lane (Claude, with a rule-based safety net)."""
import json
from common import ask_claude, parse_json

ROUTES = {
    "fast_track": "Under $5,000, simple, no injuries",
    "standard_review": "$5,000 to $50,000, or anything needing documents checked",
    "specialist_review": "Over $50,000, injuries, legal elements or unclear cause",
}


def route_by_amount(facts):
    amount = facts.get("amount") or 0
    if facts.get("injuries") or amount > 50_000:
        return "specialist_review"
    return "standard_review" if amount >= 5_000 else "fast_track"


def route_claim(claim_text, facts):
    prompt = f"""You are a claim routing specialist. Pick ONE route.
Routes: {json.dumps(ROUTES)}
Return ONLY JSON: {{"route": "<route name>", "reasoning": "one sentence"}}

Claim: {claim_text}
Facts: {json.dumps(facts)}"""
    answer = parse_json(ask_claude(prompt, 200)) or {}
    if answer.get("route") in ROUTES:
        return {"route": answer["route"], "reasoning": answer.get("reasoning", ""), "source": "claude"}
    return {"route": route_by_amount(facts), "source": "rule",
            "reasoning": "Fallback: Claude's answer was not a valid route"}

In [ ]:
router = load("router")
route = router.route_claim(sample_claim, sample_facts)
print(json.dumps(route, indent=2))

## Step 10: Pattern 3, RAG (policy-grounded answer)
Retrieves the policy text for the claim type, then answers using only that text.
The knowledge base is a small dictionary here; in production use Amazon Bedrock Knowledge Bases or a vector store.

In [ ]:
%%writefile deploy/rag.py
"""Pattern 3 - RAG: retrieve the policy text for the claim type, then answer using only that text.

The knowledge base is a small dictionary (simulated retrieval). In production, swap it for
Amazon Bedrock Knowledge Bases or a vector store over real policy documents.
"""
from common import ask_claude

POLICY_KB = {
    "auto": {"coverage_limit": 25_000, "deductible": 500,
             "text": "Auto policy covers collision damage up to $25,000 per incident, with a $500 "
                     "deductible. Claims must be filed within 30 days of the incident."},
    "property": {"coverage_limit": 75_000, "deductible": 1_000,
                 "text": "Property policy covers sudden water damage (e.g. a burst pipe) and storm damage, "
                         "excluding gradual leaks or neglect. Coverage cap is $75,000 per incident, "
                         "$1,000 deductible."},
    "health": {"coverage_limit": 150_000, "deductible": 250,
               "text": "Health policy covers medically necessary procedures up to $150,000. "
                       "Procedures over $10,000 need pre-authorisation."},
    "liability": {"coverage_limit": 1_000_000, "deductible": 2_500,
                  "text": "Liability policy covers third-party injury and damage up to $1,000,000. "
                          "Incidents must be reported within 72 hours; legal review over $50,000."},
}
GENERAL = {"coverage_limit": 10_000, "deductible": 500,
           "text": "No specific policy on file: general underwriting rules apply, limit $10,000."}


def retrieve_policy(claim_type):
    """RETRIEVE: look up the policy for this claim type."""
    return POLICY_KB.get(claim_type, GENERAL)


def rag_answer(claim_text, claim_type):
    """GENERATE: answer using only the retrieved policy text."""
    policy = retrieve_policy(claim_type)
    prompt = f"""Relevant policy text: {policy['text']}

Claim: {claim_text}

Using ONLY the policy text above, decide approve / deny / escalate in 3-4 plain sentences,
citing the policy terms that apply. No markdown."""
    return ask_claude(prompt, 400)

In [ ]:
rag = load("rag")
print("Retrieved policy:", rag.retrieve_policy(sample_facts["claim_type"])["text"])
print()
print(rag.rag_answer(sample_claim, sample_facts["claim_type"]))

## Step 11: Pattern 7, Tool use
Claude is given two tools. It decides which to call; Python runs them and returns the results to Claude.

In [ ]:
%%writefile deploy/tools.py
"""Pattern 7 - Tool use: Claude decides when to call our Python functions."""
import json
from common import get_client, MODEL

AVERAGE_REPAIR_COST = {"rear-end collision": 5_000, "fender bender": 1_500, "water damage": 15_000,
                       "storm damage": 20_000, "roof damage": 20_000, "theft": 8_000}


def lookup_average_repair_cost(damage_type):
    """Simulated industry database of average repair costs."""
    key = next((k for k in AVERAGE_REPAIR_COST if k in str(damage_type).lower()), None)
    return {"damage_type": damage_type, "average_cost": AVERAGE_REPAIR_COST.get(key, 5_000),
            "matched": key or "default"}


def estimate_fraud_risk(amount, prior_claims=0, policy_age_months=None):
    """Simulated fraud score from simple, explainable signals (whole points, no rounding surprises)."""
    points, flags = 10, []
    if (prior_claims or 0) >= 3:
        points += 30
        flags.append("3 or more prior claims")
    if policy_age_months is not None and policy_age_months < 3:
        points += 20
        flags.append("policy younger than 3 months")
    if (amount or 0) > 50_000:
        points += 20
        flags.append("large amount")
    level = "high" if points >= 50 else "medium" if points >= 30 else "low"
    return {"risk_points": points, "risk_level": level, "flags": flags}


TOOL_FUNCTIONS = {"lookup_average_repair_cost": lookup_average_repair_cost,
                  "estimate_fraud_risk": estimate_fraud_risk}

TOOLS = [
    {"name": "lookup_average_repair_cost",
     "description": "Get the average repair cost for a type of damage, e.g. 'rear-end collision'.",
     "input_schema": {"type": "object", "properties": {"damage_type": {"type": "string"}},
                      "required": ["damage_type"]}},
    {"name": "estimate_fraud_risk",
     "description": "Score fraud risk from the claim amount, number of prior claims and policy age in months.",
     "input_schema": {"type": "object",
                      "properties": {"amount": {"type": "number"}, "prior_claims": {"type": "integer"},
                                     "policy_age_months": {"type": "number"}},
                      "required": ["amount"]}},
]


def _as_dict(block):
    if block.type == "tool_use":
        return {"type": "tool_use", "id": block.id, "name": block.name, "input": block.input}
    return {"type": "text", "text": block.text}


def run_tool_loop(prompt, max_turns=6):
    """Let Claude call tools until it gives a final answer. Returns (final_text, tool_log)."""
    client, messages, log = get_client(), [{"role": "user", "content": prompt}], []
    for _ in range(max_turns):
        response = client.messages.create(model=MODEL, max_tokens=800,
                                          tools=TOOLS, messages=messages)
        if response.stop_reason != "tool_use":
            return "".join(b.text for b in response.content if b.type == "text"), log
        messages.append({"role": "assistant", "content": [_as_dict(b) for b in response.content]})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                func = TOOL_FUNCTIONS.get(block.name)
                output = func(**block.input) if func else {"error": "unknown tool"}
                log.append({"tool": block.name, "input": block.input, "output": output})
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": json.dumps(output)})
        messages.append({"role": "user", "content": results})
    return "Stopped: too many tool calls.", log


def tool_agent(claim_text):
    return run_tool_loop(
        f"Claim: {claim_text}\n\nUse the tools to look up the average repair cost and the fraud risk, "
        "then say in 2-3 plain sentences whether the claimed amount is reasonable.")

In [ ]:
tools = load("tools")
answer, tool_log = tools.tool_agent(sample_claim)
for call in tool_log:
    print(f"Claude called {call['tool']}({call['input']}) -> {call['output']}")
print()
print(answer)

## Step 12: Pattern 4, Guardrails
Fixed business rules, checked in Python after the AI decides. Here a $95,000 roof claim is proposed as
"approve" and the guardrails block it.

In [ ]:
%%writefile deploy/guardrails.py
"""Pattern 4 - Guardrails: fixed business rules checked in code, after the AI decides.

Claude extracted the facts; plain Python enforces the rules, so a rule always gives
the same answer and cannot be argued away by the claim text.
"""

GUARDRAIL_RULES = [
    "Claims over $50,000 are never auto-approved.",
    "Claims must state the damage amount and the cause.",
    "Policies younger than 1 month need manual review.",
    "The amount must be within the policy's coverage limit.",
]


def check_guardrails(facts, decision, policy):
    amount = facts.get("amount")
    age = facts.get("policy_age_months")
    checks = {
        "amount_and_cause_stated": amount is not None and bool(facts.get("cause_stated")),
        "within_coverage_limit": amount is not None and amount <= policy["coverage_limit"],
        "policy_older_than_1_month": age is None or age >= 1,
        "no_large_auto_approval": not (decision == "approve" and (amount or 0) > 50_000),
    }
    failed = [name for name, ok in checks.items() if not ok]
    return {"passed": not failed, "failed": failed, "checks": checks,
            "final_decision": decision if not failed else "escalate"}

In [ ]:
guardrails = load("guardrails")
big_claim = "Roof collapse after a storm, $95,000 repair estimate, policy active for 3 years."
big_facts = facts_mod.extract_facts(big_claim)
print("Rules:", *guardrails.GUARDRAIL_RULES, sep="\n  - ")
print()
for claim_text, f in [(sample_claim, sample_facts), (big_claim, big_facts)]:
    result = guardrails.check_guardrails(f, "approve", rag.retrieve_policy(f["claim_type"]))
    print(claim_text)
    print("  PASS" if result["passed"] else f"  FAIL: {result['failed']}", "-> final:", result["final_decision"])

## Step 13: Pattern 6, Reflection
A second Claude call audits a weak decision and returns an improved one.

In [ ]:
%%writefile deploy/reflection.py
"""Pattern 6 - Reflection: a second Claude call audits the decision and proposes a better one."""
import json
from common import ask_claude, parse_json

REVISE_BELOW = 7   # the pipeline adopts the revised decision when the rating is below this


def reflect_on_decision(claim_text, decision):
    prompt = f"""You are a senior claims auditor.
Claim: {claim_text}
Decision to review: {json.dumps(decision)}

Critique it: is anything missing, inconsistent or unjustified? Then give your improved decision.
Return ONLY JSON: {{"rating": 1-10, "critique": "...",
"revised_decision": "approve | deny | escalate", "revised_confidence": 0.0-1.0,
"revised_reasoning": "..."}}"""
    result = parse_json(ask_claude(prompt, 600))
    if not result:
        return {"rating": None, "critique": "Reply was not valid JSON",
                "revised_decision": decision.get("decision")}
    result["revised_decision"] = str(result.get("revised_decision", "")).lower()
    return result

In [ ]:
reflection = load("reflection")
weak = {"decision": "approve", "confidence": 0.6, "reasoning": "Looks fine."}
water_claim = "Water damage from a burst pipe, $12,000 estimated repair, policy 6 months old."
review = reflection.reflect_on_decision(water_claim, weak)
print(json.dumps(review, indent=2))

## Step 14: Pattern 5, Human escalation
Clear rules for when a person must review the claim.

In [ ]:
%%writefile deploy/escalation.py
"""Pattern 5 - Human escalation: clear rules for when a person must review the claim."""


def needs_human(decision, guard, facts, duplicate=False):
    reasons = []
    if decision.get("confidence", 0) < 0.75:
        reasons.append(f"Low AI confidence ({decision.get('confidence', 0):.0%})")
    if decision.get("risk_level") == "high":
        reasons.append("High fraud/risk level")
    if not guard["passed"]:
        reasons.append("Guardrail failed: " + ", ".join(guard["failed"]))
    if (facts.get("prior_claims") or 0) >= 3:
        reasons.append(f"{facts['prior_claims']} prior claims")
    if facts.get("injuries"):
        reasons.append("Injuries involved")
    if duplicate:
        reasons.append("Same claim text was already submitted")
    return {"escalate": bool(reasons), "reasons": reasons}

In [ ]:
escalation = load("escalation")
frequent_claim = "Minor fender bender, $800 damage, 5-year policy holder, 3 prior claims in 2 years."
frequent_facts = facts_mod.extract_facts(frequent_claim)
confident = {"decision": "approve", "confidence": 0.9, "risk_level": "low"}
ok_guard = {"passed": True, "failed": []}
print("Sample claim:  ", escalation.needs_human(confident, ok_guard, sample_facts))
print("Frequent claim:", escalation.needs_human(confident, ok_guard, frequent_facts))

## Step 15: Full pipeline, all 8 patterns chained
Memory check, facts, plan, route, policy (RAG), tool-assisted decision, reflection, guardrails, escalation,
then save to S3. Claim 5 repeats claim 1's text, so memory flags it as a duplicate.
Each claim takes about 30 to 60 seconds.

In [ ]:
%%writefile deploy/pipeline.py
"""Full pipeline: every claim goes through all 8 patterns. The EC2 web app imports this file."""
import json

from facts import extract_facts
from planner import plan_claim
from router import route_claim
from rag import retrieve_policy
from tools import run_tool_loop
from reflection import reflect_on_decision, REVISE_BELOW
from guardrails import check_guardrails
from escalation import needs_human
from memory import S3ClaimMemory, text_hash
from common import parse_json


def decide(claim_text, facts, plan, route, policy):
    """Pattern 7 + decision: Claude uses the tools, then returns a JSON decision."""
    prompt = f"""You are the final decision agent for an insurance claim.
Claim: {claim_text}
Facts: {json.dumps(facts)}
Plan: {json.dumps(plan['steps'])}
Route: {route['route']}
Policy (retrieved): {policy['text']}

Use the tools to check the repair cost and fraud risk, then return ONLY JSON:
{{"decision": "approve | deny | escalate", "confidence": 0.0-1.0,
"risk_level": "low | medium | high", "reasoning": "2-3 sentences citing the policy and tool results"}}"""
    text, tool_log = run_tool_loop(prompt)
    d = parse_json(text) or {}
    decision = {
        "decision": str(d.get("decision", "escalate")).lower(),
        "risk_level": str(d.get("risk_level", "medium")).lower(),
        "reasoning": d.get("reasoning", "Decision reply was not valid JSON."),
    }
    try:
        decision["confidence"] = float(d.get("confidence", 0.5))
    except (TypeError, ValueError):
        decision["confidence"] = 0.5
    return decision, tool_log


def run_full_pipeline(claim_id, claim_text, memory=None):
    memory = memory or S3ClaimMemory()
    steps = []

    duplicate_of = memory.find_duplicate(claim_text, claim_id)          # 8 memory (read)
    facts = extract_facts(claim_text)
    steps.append(f"Facts: {facts['claim_type']}, amount {facts['amount']}")
    plan = plan_claim(claim_text, facts)                                 # 1 planner
    steps.append(f"Plan: {len(plan['steps'])} steps")
    route = route_claim(claim_text, facts)                               # 2 router
    steps.append(f"Route: {route['route']} ({route['source']})")
    policy = retrieve_policy(facts["claim_type"])                        # 3 RAG
    steps.append(f"Policy: limit ${policy['coverage_limit']:,}")
    decision, tool_log = decide(claim_text, facts, plan, route, policy)  # 7 tools + decision
    steps.append(f"Decision: {decision['decision']} ({decision['confidence']:.0%}), "
                 f"{len(tool_log)} tool calls")
    original = dict(decision)

    reflection = reflect_on_decision(claim_text, decision)               # 6 reflection
    rating = reflection.get("rating")
    revised = isinstance(rating, (int, float)) and rating < REVISE_BELOW
    if revised:
        decision["decision"] = reflection.get("revised_decision") or decision["decision"]
        decision["confidence"] = float(reflection.get("revised_confidence") or decision["confidence"])
        decision["reasoning"] = reflection.get("revised_reasoning", decision["reasoning"])
    steps.append(f"Reflection: {rating}/10, {'revised' if revised else 'kept'}")

    guard = check_guardrails(facts, decision["decision"], policy)        # 4 guardrails
    steps.append("Guardrails: " + ("passed" if guard["passed"] else "FAILED " + ", ".join(guard["failed"])))
    escalation = needs_human(decision, guard, facts, bool(duplicate_of)) # 5 escalation
    final = "escalate" if escalation["escalate"] else guard["final_decision"]
    steps.append("Escalation: " + ("; ".join(escalation["reasons"]) or "not needed"))

    result = {
        "claim_id": claim_id, "claim_text": claim_text, "text_hash": text_hash(claim_text),
        "final_decision": final, "confidence": decision["confidence"],
        "risk_level": decision["risk_level"], "reasoning": decision["reasoning"],
        "facts": facts, "plan": plan, "route": route, "policy": policy["text"],
        "tool_calls": tool_log, "original_decision": original, "reflection": reflection,
        "decision_revised": revised, "guardrails": guard, "escalation": escalation,
        "duplicate_of": duplicate_of, "steps": steps,
    }
    memory.save_claim(claim_id, result)                                  # 8 memory (write)
    return result

In [ ]:
pipeline = load("pipeline")
claims_batch = [
    ("claim_101", sample_claim),
    ("claim_102", water_claim),
    ("claim_103", frequent_claim),
    ("claim_104", big_claim),
    ("claim_105", sample_claim),   # same text as claim_101
]
results = []
for claim_id, text in claims_batch:
    r = pipeline.run_full_pipeline(claim_id, text, mem)
    results.append(r)
    print(f"--- {claim_id}: {r['final_decision'].upper()} ({r['confidence']:.0%}, risk {r['risk_level']})")
    for line in r["steps"]:
        print("    ", line)
print()
print("Claims stored in S3:", mem.list_claims())

# Part 3: Deploy the web app to EC2

**Restarted the kernel?** Part 3 and Part 4 only need the names set in Step 2. Run the Step 2 cell once, then continue here.
You do not need to rerun Steps 3 to 16 or rebuild an image that is already in ECR.

## Step 16: Write the web app and container files
The Streamlit app imports `pipeline.py`, so the deployed app runs all 8 patterns.
Only the `deploy/` folder is sent to the build.

In [ ]:
%%writefile deploy/app.py
"""Streamlit web app on EC2: runs the full 8-pattern pipeline and shows claim history from S3."""
import uuid

import streamlit as st
from common import MODEL
from memory import S3ClaimMemory
from pipeline import run_full_pipeline

st.set_page_config(page_title="Insurance Claims AI", page_icon="🏢", layout="wide")
st.title("Insurance Claim Processing System")
st.caption(f"Multi-agent pipeline · {MODEL} · history stored in Amazon S3")


@st.cache_resource
def get_memory():
    return S3ClaimMemory()


claim_text = st.text_area("Claim details", height=120,
                          placeholder="e.g. Rear-end collision, $3,200 in vehicle damage, no injuries, "
                                      "policy active for 2 years, no prior claims.")

if st.button("Analyze Claim", type="primary") and claim_text.strip():
    claim_id = "web-" + uuid.uuid4().hex[:8]
    with st.spinner("Running the 8-pattern pipeline (about 30-60 seconds)..."):
        r = run_full_pipeline(claim_id, claim_text.strip(), get_memory())

    show = {"approve": st.success, "deny": st.error}.get(r["final_decision"], st.warning)
    show(f"**Decision: {r['final_decision'].upper()}**  ·  {r['reasoning']}")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Confidence", f"{r['confidence']:.0%}")
    c2.metric("Risk level", r["risk_level"].upper())
    c3.metric("Route", r["route"]["route"])
    c4.metric("Escalated", "YES" if r["escalation"]["escalate"] else "NO")
    if r["escalation"]["reasons"]:
        st.write("Escalation reasons: " + "; ".join(r["escalation"]["reasons"]))

    with st.expander("Pipeline steps"):
        st.write(r["steps"])
    with st.expander("Facts and plan"):
        st.json({"facts": r["facts"], "plan": r["plan"]})
    with st.expander(f"Tool calls ({len(r['tool_calls'])})"):
        st.json(r["tool_calls"])
    with st.expander("Reflection"):
        st.json({"original": r["original_decision"], "reflection": r["reflection"]})
    with st.expander("Guardrails"):
        st.json(r["guardrails"])
    st.caption(f"Saved to S3 as {claim_id}.json")

with st.expander("Claim history in S3"):
    try:
        st.write(get_memory().list_claims()[-20:])
    except Exception as e:  # noqa: BLE001
        st.write(f"Could not read history: {e}")

In [ ]:
%%writefile deploy/requirements.txt
streamlit
anthropic
boto3

In [ ]:
%%writefile deploy/Dockerfile
FROM public.ecr.aws/docker/library/python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY *.py ./
EXPOSE 8501
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.headless=true"]

In [ ]:
%%writefile deploy/.dockerignore
__pycache__/
*.ipynb
.ipynb_checkpoints/

## Step 17: Build the image with CodeBuild and push it to ECR
SageMaker Studio has no Docker daemon, so `sm-docker` sends the `deploy/` folder to **AWS CodeBuild**,
which builds the image and pushes it to **Amazon ECR**. This takes about 5 to 10 minutes.
It needs the CodeBuild trust relationship on your execution role (see the lab guide).

First, check that CodeBuild is allowed to use the role this notebook runs as. If this cell shows ❌, fix the trust policy in IAM (see the lab guide, Section 7), wait 10 seconds and run it again.

In [ ]:
caller = boto3.client("sts").get_caller_identity()["Arn"]
if ":assumed-role/" not in caller:
    print("Could not detect the execution role from", caller)
else:
    EXEC_ROLE = caller.split("/")[1]
    print("This notebook runs as role:", EXEC_ROLE)
    try:
        trust = json.dumps(boto3.client("iam").get_role(RoleName=EXEC_ROLE)["Role"]["AssumeRolePolicyDocument"])
        if "codebuild.amazonaws.com" in trust:
            print("✅ CodeBuild can use this role. Run the build cell below.")
        else:
            raise SystemExit(f"❌ The trust policy of role {EXEC_ROLE} does not include codebuild.amazonaws.com. "
                             "Open IAM > Roles > this role > Trust relationships > Edit trust policy, add it, "
                             "then run this cell again.")
    except ClientError as e:
        print(f"Could not read the trust policy ({e.response['Error']['Code']}). "
              f"Check manually that role {EXEC_ROLE} trusts codebuild.amazonaws.com before building.")

In [ ]:
!cd deploy && sm-docker build . --repository insurance-claims-ui:latest

In [ ]:
ecr = boto3.client("ecr", region_name=REGION)
image = ecr.describe_images(repositoryName=ECR_REPO, imageIds=[{"imageTag": "latest"}])["imageDetails"][0]
print("Image is in ECR:", IMAGE_URI)
print("Pushed at:", image["imagePushedAt"], "| Size (MB):", round(image["imageSizeInBytes"] / 1e6))

## Step 18: Create the EC2 IAM role
The instance gets only what the app needs: pull the image from ECR, read the API key secret,
and read/write claim history in your S3 bucket. Safe to re-run.

In [ ]:
iam = boto3.client("iam")
trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
         "Principal": {"Service": "ec2.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
try:
    iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust),
                    Description="EC2 role for the insurance claims Streamlit app")
    print("Role created:", ROLE_NAME)
except iam.exceptions.EntityAlreadyExistsException:
    print("Role already exists:", ROLE_NAME)

iam.attach_role_policy(RoleName=ROLE_NAME,
                       PolicyArn="arn:aws:iam::aws:policy/AmazonEC2ContainerRegistryReadOnly")
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="AppAccess", PolicyDocument=json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "secretsmanager:GetSecretValue",
         "Resource": f"arn:aws:secretsmanager:{REGION}:{ACCOUNT_ID}:secret:{SECRET_NAME}*"},
        {"Effect": "Allow", "Action": ["s3:GetObject", "s3:PutObject"],
         "Resource": f"arn:aws:s3:::{BUCKET}/*"},
        {"Effect": "Allow", "Action": "s3:ListBucket", "Resource": f"arn:aws:s3:::{BUCKET}"},
    ]}))

try:
    iam.create_instance_profile(InstanceProfileName=ROLE_NAME)
    iam.add_role_to_instance_profile(InstanceProfileName=ROLE_NAME, RoleName=ROLE_NAME)
    print("Instance profile created")
except iam.exceptions.EntityAlreadyExistsException:
    print("Instance profile already exists")
iam.get_waiter("instance_profile_exists").wait(InstanceProfileName=ROLE_NAME)
print("IAM setup complete")

## Step 19: Create a security group that only allows your IP
Open https://checkip.amazonaws.com in a new browser tab, copy the address it shows, and paste it when prompted.
Only your computer will be able to open the app.

In [ ]:
ec2 = boto3.client("ec2", region_name=REGION)
MY_IP = input("Your public IP address (from https://checkip.amazonaws.com): ").strip()

vpcs = ec2.describe_vpcs(Filters=[{"Name": "isDefault", "Values": ["true"]}])["Vpcs"]
assert vpcs, "No default VPC in this Region. Create one (VPC console > Actions > Create default VPC), then re-run."
VPC_ID = vpcs[0]["VpcId"]

found = ec2.describe_security_groups(Filters=[{"Name": "group-name", "Values": [SG_NAME]},
                                              {"Name": "vpc-id", "Values": [VPC_ID]}])["SecurityGroups"]
if found:
    SG_ID = found[0]["GroupId"]
    print("Security group already exists:", SG_ID)
else:
    SG_ID = ec2.create_security_group(GroupName=SG_NAME, VpcId=VPC_ID,
                                      Description="Streamlit port for the insurance claims lab")["GroupId"]
    print("Security group created:", SG_ID)

try:
    ec2.authorize_security_group_ingress(GroupId=SG_ID, IpPermissions=[{
        "IpProtocol": "tcp", "FromPort": 8501, "ToPort": 8501,
        "IpRanges": [{"CidrIp": f"{MY_IP}/32", "Description": "Lab user"}]}])
    print(f"Port 8501 opened for {MY_IP} only")
except ClientError as e:
    if "Duplicate" not in str(e):
        raise
    print("Rule for this IP already exists")

## Step 20: Launch the EC2 instance
Launches a `t3.small` with the latest **Amazon Linux 2023** (looked up from AWS Systems Manager, so it works
in any Region). At start-up the instance installs Docker, pulls your image from ECR and runs the app on port 8501.

In [ ]:
user_data = f"""#!/bin/bash
dnf install -y docker
systemctl enable --now docker
aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com
docker run -d --restart always -p 8501:8501 -e AWS_REGION={REGION} -e CLAIMS_BUCKET={BUCKET} -e CLAUDE_MODEL={MODEL} {IMAGE_URI}
"""

for attempt in range(6):
    try:
        response = ec2.run_instances(
            ImageId="resolve:ssm:/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-x86_64",
            InstanceType="t3.small", MinCount=1, MaxCount=1,
            UserData=user_data,
            IamInstanceProfile={"Name": ROLE_NAME},
            SecurityGroupIds=[SG_ID],
            # IMDSv2 with 2 hops lets the Docker container use the instance role
            MetadataOptions={"HttpTokens": "required", "HttpPutResponseHopLimit": 2, "HttpEndpoint": "enabled"},
            TagSpecifications=[{"ResourceType": "instance",
                                "Tags": [{"Key": "Name", "Value": "insurance-claims-ui"}]}])
        break
    except ClientError as e:
        if "Instance Profile" in str(e) and attempt < 5:
            print("IAM role not visible to EC2 yet, retrying in 10 seconds...")
            time.sleep(10)
        else:
            raise

INSTANCE_ID = response["Instances"][0]["InstanceId"]
print("Instance launched:", INSTANCE_ID)

## Step 21: Wait for the instance, then open the app
The instance is "running" after about a minute, but Docker needs **3 to 5 more minutes** to pull and start the app.
If the page does not load, wait a minute and refresh.

In [ ]:
print("Waiting for the instance to reach the running state...")
ec2.get_waiter("instance_running").wait(InstanceIds=[INSTANCE_ID])
public_ip = ec2.describe_instances(InstanceIds=[INSTANCE_ID])["Reservations"][0]["Instances"][0]["PublicIpAddress"]
print("Instance running!")
print("The app will be ready in about 3-5 minutes. Open this URL in your browser:")
print(f"  http://{public_ip}:8501")

Try these new claims in the app (they differ from Step 15, so none is a duplicate):

| Claim text | Expected result |
| --- | --- |
| Side-swipe in a car park, $2,400 to repair the door, no injuries, policy active for 4 years, no prior claims. | APPROVE |
| Kitchen flooded after a washing machine hose burst, $9,500 repair estimate, policy active for 18 months. | Usually APPROVE |
| Bumper scratch in a supermarket car park, $600 damage, policy active for 3 years, 4 prior claims this year. | ESCALATE (4 prior claims) |
| House fire damaged two rooms, $120,000 repair estimate, policy active for 5 years. | ESCALATE (over the $75,000 limit) |

Submit the same claim text a second time and it is escalated as a duplicate (the S3 memory at work).

# Part 4: Clean up

## Step 22: Delete the AWS resources
Deletes the EC2 instance, security group, ECR repository, secret, S3 bucket, EC2 role and the CodeBuild logs.
Every step reports success or failure. If any line shows ❌, delete that resource in the console
(see the Clean-Up section of the lab guide).

In [ ]:
import boto3
from botocore.exceptions import ClientError

REGION = boto3.session.Session().region_name or "us-east-1"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
BUCKET = f"insurance-claims-history-{ACCOUNT_ID}"
ec2, ecr, iam = boto3.client("ec2", region_name=REGION), boto3.client("ecr", region_name=REGION), boto3.client("iam")
sm, logs = boto3.client("secretsmanager", region_name=REGION), boto3.client("logs", region_name=REGION)
failed = []

def step(name, fn):
    try:
        fn()
        print("✅", name)
    except Exception as e:
        failed.append(name)
        print("❌", name, "-", e)

def terminate_instances():
    ids = [i["InstanceId"] for r in ec2.describe_instances(Filters=[
        {"Name": "tag:Name", "Values": ["insurance-claims-ui"]},
        {"Name": "instance-state-name", "Values": ["pending", "running", "stopping", "stopped"]}])["Reservations"]
        for i in r["Instances"]]
    if ids:
        ec2.terminate_instances(InstanceIds=ids)
        ec2.get_waiter("instance_terminated").wait(InstanceIds=ids)

def delete_security_group():
    for sg in ec2.describe_security_groups(Filters=[{"Name": "group-name", "Values": ["insurance-claims-ui-sg"]}])["SecurityGroups"]:
        ec2.delete_security_group(GroupId=sg["GroupId"])

def delete_bucket():
    s3 = boto3.resource("s3")
    try:
        bucket = s3.Bucket(BUCKET)
        bucket.objects.all().delete()
        bucket.delete()
    except ClientError as e:
        if "NoSuchBucket" not in str(e):
            raise

def delete_role():
    for call in (lambda: iam.remove_role_from_instance_profile(InstanceProfileName="EC2InsuranceClaimsRole", RoleName="EC2InsuranceClaimsRole"),
                 lambda: iam.delete_instance_profile(InstanceProfileName="EC2InsuranceClaimsRole"),
                 lambda: iam.detach_role_policy(RoleName="EC2InsuranceClaimsRole", PolicyArn="arn:aws:iam::aws:policy/AmazonEC2ContainerRegistryReadOnly"),
                 lambda: iam.delete_role_policy(RoleName="EC2InsuranceClaimsRole", PolicyName="AppAccess"),
                 lambda: iam.delete_role(RoleName="EC2InsuranceClaimsRole")):
        try:
            call()
        except iam.exceptions.NoSuchEntityException:
            pass

def delete_ecr():
    try:
        ecr.delete_repository(repositoryName="insurance-claims-ui", force=True)
    except ecr.exceptions.RepositoryNotFoundException:
        pass

def delete_secret():
    try:
        sm.delete_secret(SecretId="insurance-claims/anthropic-api-key", ForceDeleteWithoutRecovery=True)
    except sm.exceptions.ResourceNotFoundException:
        pass

def delete_build_logs():
    for g in logs.describe_log_groups(logGroupNamePrefix="/aws/codebuild/sagemaker-studio")["logGroups"]:
        logs.delete_log_group(logGroupName=g["logGroupName"])

step("EC2 instance terminated", terminate_instances)
step("Security group deleted", delete_security_group)
step("ECR repository deleted", delete_ecr)
step("Secret deleted", delete_secret)
step("S3 bucket deleted", delete_bucket)
step("EC2 IAM role deleted", delete_role)
step("CodeBuild log groups deleted", delete_build_logs)

print()
print("🧹 Cleanup complete!" if not failed else f"⚠️ Cleanup finished with {len(failed)} error(s): {failed}. Delete these in the console.")